In [2]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu118
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO("yolo11n.pt")

In [46]:
cap = cv2.VideoCapture(0)
paused = False
fullImg = None
key = cv2.waitKey(1)

while(True):
  key = cv2.waitKey(1)

  if(key == 27):
    break
  elif(key == ord('p') or key == ord('P')):
    paused = not paused

  if(not(fullImg is None)):
    cv2.imshow("img", fullImg)
    if(key == ord('s') or key == ord('S')):
      cv2.imwrite("fullImg.png", fullImg)
  
  if(paused):
    continue

  bCap, img = cap.read()
  if(not bCap):
    break

  img = cv2.resize(img, None, fx=0.5, fy=0.5)
  img = cv2.flip(img, 1)

  results = model(img, stream=True, verbose=False)

  imgYolo = img.copy()

  for r in results:
    imgYolo = r.plot(img=imgYolo)

  imgGr = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  _,imgThr = cv2.threshold(img, 96, 255, cv2.THRESH_BINARY)

  imgEdg = cv2.Canny(imgGr, 50, 100)

  imgGr = cv2.cvtColor(imgGr, cv2.COLOR_GRAY2BGR)
  imgEdg = cv2.cvtColor(imgEdg, cv2.COLOR_GRAY2BGR)

  hor1 = np.concatenate((img,imgThr,imgYolo), axis=1)
  hor2 = np.concatenate((imgGr,imgEdg,img), axis=1)

  fullImg = np.concatenate((hor1, hor2), axis=0)



cap.release()
cv2.destroyAllWindows()